# Coelho: mixture prefit, fixed mask, and Gaussian GP/Vecchia inference

Fit two IRD orders using this sequence:

1. Build the observation/grid and estimate RV with the CCF.
2. Run a Gaussian-mixture SVI prefit with explicit continuum coefficients.
3. Inspect mismatch probabilities, then fix a velocity-dilated mask.
4. Fit a separate Gaussian GP/Vecchia model with analytic continuum
   marginalization and sample it with NUTS.

The prefit and final model share physical settings. The final model has no
mixture parameters or sampled continuum coefficients. `GP_SOLVER="vecchia1"`
selects the approximate nearest-neighbor likelihood; use `"auto"` for the
exact Matérn-3/2 GP solver selected for the current device.

## Setup

From the repository root install the source checkout into the notebook kernel:

```bash
python -m pip install -e . seaborn ipykernel
```

Run from the repository root or `examples/specfit`. The data and local Coelho
`*_normed.npz` grids are read from `examples/specfit/coelho/`. The setup selects
CPU, enables JAX x64, and requests one host device for one NUTS chain.

Choose `result_dir` before running. Figures are saved there as PDFs;
`save_fit_result(..., overwrite=False)` refuses to replace an existing bundle.
Run the cells in order to generate the diagnostics.


In [ ]:
import numpyro 
num_chains = 1
numpyro.set_host_device_count(num_chains)

from pathlib import Path
import sys

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src/jaxstar/specfit/model.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from the jaxstar repository.")
sys.path.insert(0, str(ROOT / "src"))

import jax
jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from numpyro.infer import init_to_value
from jaxstar.specfit import (
    Observation, SpecFit, select_grid_paths,
)

from jaxstar.specfit.numpyro_model import make_model_single

plt.rcParams.update({
    "figure.figsize": (10, 4), "figure.dpi": 100, "figure.facecolor": "white",
})
print("Backend:", jax.default_backend(), "JAX:", jax.__version__)

In [ ]:
# Separate output directory for this workflow
result_dir = ROOT / "examples/specfit/results_mixture_prefit"
result_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
import seaborn as sns
sns.set(style='ticks', font_scale=1.3, font='times')
from matplotlib import rc
rc('text', usetex=False)

## 1. Load the observed spectrum

The CSV contains concatenated orders with 2,048 pixels per order. Reshape each
column and select the zero-based order indices `[8, 9]`. These indices also
become the region labels used in diagnostics and output filenames.

Convert `lam` from nm to Å, and read `normed_flux`, `flux_error`, and `all_mask`.
In an `Observation`, `mask=True` excludes a pixel. The loader adds nonfinite
flux/error values and nonpositive errors to this fixed data mask. Wavelengths
must be finite, positive, and strictly increasing within every order, including
masked pixels.

Inspect the usable-pixel counts and the plot before fitting.

In [ ]:
gridpath = ROOT / "examples/specfit/coelho"
datapath = gridpath / "sample_data_ird.csv"
orders = [8, 9]


def load_ird_spectrum(datapath, orders):
    data = pd.read_csv(datapath)

    def rows(name):
        return data[name].to_numpy().reshape(-1, 2048)[orders].copy()

    wavelength = rows("lam") * 10.0  # nm -> Angstrom
    flux = rows("normed_flux")
    error = rows("flux_error")
    mask = rows("all_mask").astype(bool)
    mask |= ~np.isfinite(flux) | ~np.isfinite(error) | (error <= 0)
    return wavelength, flux, error, mask


wav_obs, flux_obs, error_obs, mask_obs = load_ird_spectrum(datapath, orders)
obs = Observation(
    wav_obs, flux_obs, error_obs, mask=mask_obs, region=orders,
)
print("Observation shape:", obs.shape)
print("Usable pixels per order:", np.sum(obs.valid, axis=1))

In [ ]:
fig, axes = plt.subplots(obs.n_regions, 1, figsize=(10, 3 * obs.n_regions), squeeze=False)
for i, region in enumerate(obs.region):
    ax = axes[i, 0]
    usable = ~mask_obs[i]
    ax.plot(wav_obs[i], flux_obs[i], color="0.8", lw=0.7, label="all pixels")
    ax.plot(wav_obs[i, usable], flux_obs[i, usable], ".", ms=2, label="usable")
    ax.set(title=f"Region {region}", xlabel="Wavelength [Å]", ylabel="Normalized flux")
    ax.legend()
fig.tight_layout()

## 2. Inspect the grid selection

`select_grid_paths` finds one `*_normed.npz` file for each observed region. It
checks the wavelength range stored in each file's `wavgrid` and requires at
least `wav_margin=3.0` Å of coverage beyond both observed endpoints, including
masked pixels. A missing or ambiguous match raises an error identifying the
region and candidate files.

The table shows which grid will be used for each order. This inspection is
optional: `SpecFit.from_grid` performs the same selection in the next section.
The NPZ files do not specify an air/vacuum wavelength convention, and no
conversion is applied. When substituting other data, use matching wavelength
units and conventions for the observations and grids.

In [ ]:
grid_paths = select_grid_paths(
    gridpath, obs.wavelength, regions=obs.region,
    wav_margin=3.0, pattern="*_normed.npz",
)
display(pd.DataFrame({"region": obs.region, "grid": [p.name for p in grid_paths]}))

## 3. Build the spectral model and fitter

`SpecFit.from_grid` loads the selected grids, resamples them onto grids evenly
spaced in log wavelength, and combines a `SpecModel` with the `Observation`.
`flux_kind="normalized"` declares that the stored grid spectra are normalized.

Here, `pixels=5000` sets the number of model wavelength samples in each region.
For these grids, the resulting velocity spacing is about 0.82 km/s. This is the
model sampling, separate from the observed pixel count and instrumental
resolving power. `vmax=50.0` km/s sets the half-width of the broadening kernel.

The following table reports the available atmosphere-parameter bounds; the
printed shapes and velocity steps describe the prepared model. If a model has
already been constructed, it can also be paired with data using
`SpecFit(obs, specmodel)`.

In [ ]:
sf = SpecFit.from_grid(
    gridpath, obs, library="coelho", pattern="*_normed.npz",
    wav_margin=3.0, pixels=5000, vmax=50.0, flux_kind="normalized",
)

In [ ]:
specmodel = sf.specmodel
spectra = specmodel.spectra

axis_bounds = {
    name: (float(spectra.grid.axis(name)[0]), float(spectra.grid.axis(name)[-1]))
    for name in spectra.grid.axis_names
}
display(pd.DataFrame(axis_bounds, index=["lower", "upper"]).T)
print("Model wavelength shape:", spectra.wavelength.shape)
print("Model velocity steps [km/s]:",
      np.asarray(specmodel.velocity_grid[:, 1] - specmodel.velocity_grid[:, 0]))

## 4. Estimate radial velocity with the CCF

Cross-correlate the usable observed pixels against a template with
`teff=6000` K, `logg=4.0`, `feh=0.0`, and `alpha=0.0`. `v_limit=100.0` restricts
the peak search to velocities within ±100 km/s, while `ccfvmax=100.0` sets the
half-width of the combined CCF window around the estimated radial velocity.

`check_ccf` returns the median of the per-order radial velocities and the full
width at half maximum of the combined CCF, both in km/s. It also stores these
results in `sf.ccf` for constructing the priors below. The CCF width helps set a
rotation prior; it is not a direct measurement of `vsini`.

Inspect the peak and width in the plot, which is saved as `ccf.pdf`.

In [ ]:
template_atmosphere = {"teff": 6000.0, "logg": 4.0, "feh": 0.0, "alpha": 0.0}

ccf_rv, ccf_width = sf.check_ccf(
    template_atmosphere, v_limit=100.0, ccfvmax=100.0, plot=True,
    save_path=result_dir / "ccf.pdf",
)

## 5. Share physical settings between prefit and final inference

Both models use resolving power 70,000 per order, the same atmosphere bounds,
CCF-derived RV/rotation priors and physical gravity ceiling. Keep any physical
prior or `fixed` overrides in `shared_model_kwargs` so both stages receive them.

The prefit always has no GP and explicit fourth-degree continuum coefficients.
The final model uses `GP_SOLVER` and analytically marginalized continuum with
the same Gaussian coefficient priors (constant scale 0.1, higher-order 0.05).
Jitter has the same default LogUniform(0.001, 0.05) prior in both stages.


In [ ]:
import numpyro.distributions as dist

RESOLVING_POWER = 70_000.0
USE_EMPIRICAL_VMACRO = False
GP_SOLVER = "vecchia1"  # "auto" for the exact GP
USE_GP = True

shared_model_kwargs = dict(
    resolving_power=np.full(obs.n_regions, RESOLVING_POWER),
    use_physical_logg_max=True,
    use_empirical_vmic=False, use_empirical_vmacro=USE_EMPIRICAL_VMACRO,
)
prefit_kwargs = dict(
    shared_model_kwargs,
    # mismatch_scale=dist.HalfNormal(0.2),  # Optional prior override
    #fixed={'logg': 4.}
)


## 6. Run and inspect the mixture prefit

The helper supplies `mismatch_fraction ~ Beta(1, 9)` and
`mismatch_scale ~ HalfNormal(0.1)` by default. These are priors, separate
from starting values: continuum `[1, 0, ...]`, jitter 0.002, scale 0.1 and
fraction 0.01. Optional `p_initial` entries take precedence; the remaining
parameters are drawn from their conditional priors.

The helper returns a model and a point without changing masks. A fresh fitter
has no additional fit mask. To redo prefit from the original observation mask
later, explicitly call `sf.reset_fit_mask()` first.


In [ ]:
prefit_model, prefit_point = sf.run_mixture_prefit(
    rng_key=jax.random.PRNGKey(0), model_kwargs=prefit_kwargs,
    step_size=1e-2, num_steps=2000,
    p_initial={'logg': 4.},
)
display(prefit_model.check_prior())
prefit_point


In [ ]:
#prefit_prediction = sf.predict(model=prefit_model, point=prefit_point)
#mismatch_probability = np.asarray(prefit_prediction["mismatch_probability"])
_ = sf.plot_models(
    model=prefit_model, point=prefit_point, show_mismatch=True,
    save_path=result_dir / "prefit.pdf",
)
plt.show()


## 7. Build the mask once from the prefit probabilities

Use `q > 0.5` as seeds and dilate by 10 km/s to include neighboring line wings.
`mask_outliers` consumes the model's probabilities without fitting another
residual mixture. Fixed data masks are preserved. Inference excludes both
masks; prefit probabilities can still be displayed at fit-masked pixels.

Review the diagnostics and mask summary here. Keep this mask fixed throughout
the final SVI and NUTS calls below; no further outlier selection is performed.


In [ ]:
P_OUTLIER = 0.5
MASK_VELOCITY = 10.  # km/s

sf.mask_outliers(
    model=prefit_model, point=prefit_point, method="gaussian_mixture",
    p_outlier=P_OUTLIER, mask_velocity=MASK_VELOCITY,
    plot=True, save_path=result_dir / "outliers.pdf",
)
display(pd.DataFrame(sf.mask_summary()))
fixed_fit_mask = sf.fit_mask.copy()
plt.show()


## 8. Construct the final Gaussian model and obtain its initial point

The final likelihood uses a Matérn-3/2 GP (or its Vecchia approximation) and
analytic continuum marginalization. It has no mismatch parameters or sampled
continuum coefficients. The prefit's physical and jitter estimates initialize
shared sampled sites via `p_initial=prefit_point`. Entries absent from the
final model are ignored; new GP sites use NumPyro's `init_to_uniform` fallback
in unconstrained coordinates. There are no continuum coefficients to initialize
in this model. The following SVI uses the selected mask and provides a point
for NUTS.


In [ ]:
model = make_model_single(
    sf, **shared_model_kwargs, likelihood="gaussian", use_gp=USE_GP,
    gp_solver=GP_SOLVER,
)
display(model.check_prior())


In [ ]:
prefit_point

In [ ]:
p_fit = sf.run_svi(
    model=model, rng_key=jax.random.PRNGKey(1), p_initial=prefit_point,
    step_size=1e-2, num_steps=2000,
)

In [ ]:
_ = sf.plot_models(
    model=model, point=p_fit, show_gp=USE_GP, show_masked=True,
    save_path=result_dir / "final_svi.pdf",
)
display(pd.DataFrame(sf.residual_diagnostics(model=model, point=p_fit)))
plt.show()


In [ ]:
p_fit

## 9. Sample the final posterior with the fixed mask

NUTS starts from the final SVI point, with one chain, 500 warmup iterations and
500 retained samples. The mask and model remain fixed. Inspect divergences,
effective sample sizes and trace plots before interpreting results; these run
lengths are illustrative.


In [ ]:
mcmc = sf.run_nuts(
    rng_key=jax.random.PRNGKey(2), model=model,
    num_warmup=500, num_samples=500, num_chains=num_chains,
    nuts_kwargs={
        "init_strategy": init_to_value(values=p_fit),
        "target_accept_prob": 0.90, "dense_mass": True, "max_tree_depth": 6,
    },
)
np.testing.assert_array_equal(sf.fit_mask, fixed_fit_mask)
mcmc.print_summary(exclude_deterministic=True)
samples = mcmc.get_samples()


## 10. Plot the final spectrum and diagnostics

The final model has no mismatch probability panel. The probabilities used for
mask selection are stored in `sf.outlier_diagnostics["mismatch_probability"]`.

Posterior spectra average conditional predictions over up to 50 draws. With
`GP_SOLVER="vecchia1"`, continuum coefficients are still analytically
marginalized, but no latent GP mean is reconstructed or overlaid. With `"auto"`,
the exact GP mean is available. Neither path simulates measurement noise.


In [ ]:
model_figures = sf.plot_models(
    samples=samples, model=model, show_gp=USE_GP, max_draws=50,
    save_path=result_dir / "model.pdf",
)
plt.show()


In [ ]:
mcmc_figures = sf.plot_mcmc_diagnostics(
    mcmc,
    corner_vars=["teff", "logg", "feh", "alpha", "vsini", "vmacro",
                 "jitter", "gp_amplitude", "gp_scale"],
    log_vars=["jitter", "gp_amplitude", "gp_scale"],
    labels={"vmacro": "zeta", "jitter": "lnsigma",
            "gp_amplitude": "lna", "gp_scale": "lnc"},
)
for name, fig in mcmc_figures.items():
    fig.savefig(result_dir / f"{name}.pdf", bbox_inches="tight")
plt.show()


## 11. Save and reload the fit results

`save_fit_result` writes a result bundle to the `result_dir` chosen at the start:

| File | Contents |
| --- | --- |
| `arrays.npz` | Point estimate, observed arrays, fixed mask, and fit mask |
| `posterior.nc` | Posterior samples and collected MCMC diagnostics |
| `result.json` | Model settings, grid provenance, software metadata, and file manifest |

Saving uses the existing point estimate and MCMC results without rerunning
inference or spectrum prediction. PDFs are written by the corresponding
plotting cells. Keep the fitter's data and mask consistent with the sampling
run until the bundle has been saved.

With `overwrite=False`, saving stops if managed result files already exist.
Use a new `result_dir` for another analysis, or set `overwrite=True` to replace
an existing bundle deliberately. To save only a point estimate, omit
`mcmc=mcmc`; no `posterior.nc` is written in that case.

In [ ]:
from jaxstar.specfit import save_fit_result, load_fit_result, plot_mcmc_diagnostics

save_fit_result(
    result_dir, sf=sf, model=model, point=p_fit, mcmc=mcmc,
    metadata={"data_path": str(datapath.relative_to(ROOT)), "orders": orders,
              "mask_source": "mixture_prefit", "p_outlier": P_OUTLIER,
              "mask_velocity": MASK_VELOCITY},
    overwrite=False,
)
print("Saved fit:", result_dir.relative_to(ROOT))


### Load the saved bundle

`load_fit_result` and the standalone `plot_mcmc_diagnostics` function work
without loading a spectral grid or constructing a fitter. In a new session,
import these functions and set `result_dir` to the saved directory before
running the following cell.

`result.point` contains the point estimate. `result.samples` combines the chain
and draw axes for use in prediction, while `result.idata` preserves them for
diagnostics. The latter is an ArviZ `InferenceData` with ArviZ 0.x or a `DataTree`
with ArviZ 1.x.

The example displays diagnostics for a smaller selection of parameters directly
from the saved posterior. It does not write another set of PDFs.

In [ ]:
result = load_fit_result(result_dir)
p_saved = result.point
samples_saved = result.samples
idata_saved = result.idata
print("Posterior dimensions:", dict(idata_saved.posterior.sizes))

saved_diagnostics = plot_mcmc_diagnostics(
    result, corner_vars=["teff", "logg", "feh", "vsini"],
)
plt.show()


### Reconstruct spectra from saved samples

The bundle contains observations, masks, and descriptive model/grid metadata,
but not the grid arrays or an executable model. In this session, reuse
`sf.specmodel` and `model`, restore the saved fit mask, and predict from the
saved samples:

```python
restored_sf = SpecFit(result.observation, sf.specmodel)
restored_sf.set_fit_mask(result.fit_mask)
restored_figures = restored_sf.plot_models(
    samples=result.samples, model=model, show_gp=USE_GP, max_draws=50,
)
plt.show()
```

In a new session, first load the same grid and construct a model with the saved
settings, including the resolved prior bounds. `result.metadata` records these
settings and the grid information for reference; `load_fit_result` does not
rebuild the model automatically.